In [1]:
import os

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
# 下列两个任选一个
#os.environ['TRANSFORMERS_CACHE'] = '/mnt/workspace/cache/huggingface/hub'
os.environ['XDG_CACHE_HOME'] = '/mnt/workspace/cache'

In [2]:
from transformers import GPT2Tokenizer, AutoModelForCausalLM, GPT2LMHeadModel, PreTrainedTokenizer, GPT2Config 
from datasets import load_dataset
import torch
import torch.nn as nn
import numpy as np
import copy

/usr/local/lib/python3.11/site-packages/torch/cuda/__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-01-30 21:57:32.441866: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-30 21:57:34.002730: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## 模型迁移训练

### 加载原始数据

In [3]:
## 加载自定义文本文件（每行一个样本）
dataset = load_dataset(
    "text", 
    data_dir="./datas", 
    data_files={
        "train": "train_words.txt", 
        "validation": "eval_words.txt"
        
        #"train": "train_words_min.txt", 
        #"validation": "eval_words_min.txt"
    }
)
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 203069
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 50785
    })
})
{'text': 'aa'}


In [4]:
## 长度过滤
dataset = dataset.filter(lambda e: len(e['text']) <= 20)
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 202811
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 50730
    })
})

### 模型迁移-恢复分词器及模型

In [5]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2默认无pad_token，用eos_token代替

model = AutoModelForCausalLM.from_pretrained("gpt2")
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

#### 重新构建分词器

In [6]:
tokens = ['<pad>', '<|endoftext|>']
for token in list("qazwsxedcrfvtgbyhnujmikolp"):
    tokens.append(token)
# 获取token和id之间的映射mapping
token2id = {}
id2token = {}
oriid2newid = []
for new_id, token in enumerate(tokens):
    ori_id = tokenizer._convert_token_to_id(token)
    token2id[token] = new_id
    id2token[new_id] = token
    oriid2newid.append([ori_id, new_id])
print(f"映射mapping:{token2id}")
print(f"新旧id对应关系:{oriid2newid}")

映射mapping:{'<pad>': 0, '<|endoftext|>': 1, 'q': 2, 'a': 3, 'z': 4, 'w': 5, 's': 6, 'x': 7, 'e': 8, 'd': 9, 'c': 10, 'r': 11, 'f': 12, 'v': 13, 't': 14, 'g': 15, 'b': 16, 'y': 17, 'h': 18, 'n': 19, 'u': 20, 'j': 21, 'm': 22, 'i': 23, 'k': 24, 'o': 25, 'l': 26, 'p': 27}
新旧id对应关系:[[50256, 0], [50256, 1], [80, 2], [64, 3], [89, 4], [86, 5], [82, 6], [87, 7], [68, 8], [67, 9], [66, 10], [81, 11], [69, 12], [85, 13], [83, 14], [70, 15], [65, 16], [88, 17], [71, 18], [77, 19], [84, 20], [73, 21], [76, 22], [72, 23], [74, 24], [78, 25], [75, 26], [79, 27]]


In [7]:
with open("./datas/vocab.txt", "w", encoding="utf-8") as writer:
    for token_id in range(len(id2token)):
        token = id2token[token_id]
        writer.write(token + "\n")

In [ ]:

from typing import Optional, Tuple, Dict, List

from transformers import PreTrainedTokenizer, BertTokenizer


class EnglishGenTokenizer(BertTokenizer):
    def __init__(
            self,
            vocab_file,
            do_lower_case=True,
            do_basic_tokenize=True,
            never_split=None,
            unk_token="<|endoftext|>",
            sep_token="<|endoftext|>",
            pad_token="<pad>",
            cls_token="<|endoftext|>",
            mask_token="<|endoftext|>",
            eos_token='<|endoftext|>',
            bos_token="<|endoftext|>",
            tokenize_chinese_chars=True,
            strip_accents=None,
            add_eos_token=True,
            **kwargs,
    ):
        self.add_eos_token = add_eos_token
        super().__init__(
            vocab_file,
            do_lower_case=True,
            do_basic_tokenize=True,
            never_split=None,
            unk_token="<|endoftext|>",
            sep_token="<|endoftext|>",
            pad_token="<pad>",
            cls_token="<|endoftext|>",
            mask_token="<|endoftext|>",
            eos_token='<|endoftext|>',
            bos_token="<|endoftext|>",
            tokenize_chinese_chars=True,
            strip_accents=None,
            add_eos_token=add_eos_token,
            **kwargs
        )
        

    def _tokenize(self, text, **kwargs):
        """
        仅考虑英文单词的情况
        :param text:
        :param kwargs:
        :return:
        """
        tokens = []
        for token in list(text.lower()):
            if token.isalpha():
                tokens.append(token)
        #return list(text.lower())
        return tokens

    def build_inputs_with_special_tokens(
        self, token_ids_0: List[int], token_ids_1: Optional[List[int]] = None
    ) -> List[int]:
        if self.add_eos_token:
            eos_token_ids = [self.eos_token_id]
        else:
            eos_token_ids = []

        output = token_ids_0 + eos_token_ids

        if token_ids_1 is None:
            return output

        raise ValueError("异常!!!")
    
    def convert_tokens_to_string(self, tokens):
        """Converts a sequence of tokens (string) in a single string."""
        out_string = "".join(tokens).strip()
        return out_string

In [ ]:
tokenizer = EnglishGenTokenizer("./datas/vocab.txt")
print(f"新的token分词器:\n{tokenizer}")

In [ ]:
print(tokenizer("HEllo")['input_ids'])
tokenizer.decode(tokenizer("HEllo")['input_ids'], spaces_between_special_tokens=False, skip_special_tokens=True)

#### 重构模型embedding和输出参数

In [11]:
wte = model.transformer.get_input_embeddings()
new_weight = wte.weight[[t[0] for t in oriid2newid]]
new_weight = new_weight.data
num_embeddings, embedding_dim = new_weight.shape
new_wte = nn.Embedding(num_embeddings=num_embeddings, embedding_dim=embedding_dim, padding_idx=0, _weight=new_weight)
model.transformer.set_input_embeddings(new_wte)

In [12]:
## PS: 实际上可以不用重新构建，输出理论上和input embedding是相同的
lm_head = copy.deepcopy(model.lm_head)
new_lm_head_weight = lm_head.weight[[t[0] for t in oriid2newid]]
out_features, in_features = new_lm_head_weight.shape
lm_head = nn.Linear(in_features, out_features, bias=False)
lm_head.weight = nn.Parameter(new_lm_head_weight.data)
model.lm_head = lm_head

In [13]:
cfg = copy.deepcopy(model.config)
cfg.bos_token_id = tokenizer.bos_token_id
cfg.eos_token_id = tokenizer.eos_token_id
cfg.vocab_size = tokenizer.vocab_size
new_model = GPT2LMHeadModel(cfg)
new_model.load_state_dict(model.state_dict()) # 参数恢复

<All keys matched successfully>

In [14]:
model = new_model # 覆盖使用新模型
print(f"新的模型:\n{model}")

新的模型:
GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(28, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=28, bias=False)
)


### 数据处理-分词+token id转换+标签转换

In [15]:
# 对数据进行分词转换
def preprocess_function(examples):
    """
    对单个文本进行分词转换
    """
    # 分词转换
    item = tokenizer(
        examples['text'], # 对文本进行处理
        truncation=True,
        max_length=512,
        padding=False
    )
    item['labels'] = copy.deepcopy(item['input_ids'])
    del item['token_type_ids']
    return item


# 应用预处理函数（num_proc=4表示多进程加速）
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True, # 分批次处理
    num_proc=None,
    remove_columns=['text'] # 删除列
)
print(tokenized_dataset)
print(tokenized_dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 202811
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 50730
    })
})
{'input_ids': [3, 3, 1], 'attention_mask': [1, 1, 1], 'labels': [3, 3, 1]}


In [16]:
# 数据抽样以及数据分割
from datasets import DatasetDict

sample_dataset = DatasetDict({
    k: ds.take(int(len(ds) * 1.0))
    for k, ds in tokenized_dataset.shuffle(24).items()
})
sample_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 202811
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 50730
    })
})

### 构造模型评估指标

In [17]:
# 定义评估指标
import evaluate
#bleu = evaluate.load("bleu")
accuracy_metric = evaluate.load("accuracy") # 会从huggingface下载评估代码


In [18]:
def compute_metrics(eval_pred):
    # 解析预测结果和标签 (前行输出以及实际标签值)
    predictions, labels = eval_pred
    # 将 logits 转换为预测标签（取最大值索引）
    predictions = np.argmax(predictions, axis=-1)

    # 计算指标
    mask = labels != -100
    is_eqa = predictions == labels
    acc = np.sum(is_eqa) / (np.sum(mask) + 1e-8)

    # 返回指标字典（键为指标名称，值为数值）
    return {
        "acc": acc
    }

### 构造训练参数

In [20]:
from transformers import TrainingArguments, Trainer

# 可能需要安装：pip install transformers[torch]

training_args = TrainingArguments(
    output_dir="./output/gpt2-finetuned/models",  # 模型保存路径
    overwrite_output_dir=True,
    num_train_epochs=5,  # 训练轮数
    per_device_train_batch_size=128,  # 单设备训练批次大小（视GPU内存调整）
    per_device_eval_batch_size=256,   # 单设备验证批次大小
    gradient_accumulation_steps=2,  # 梯度累积（显存不足时增大，等效于增大batch_size）
    eval_strategy="epoch",    # 每轮结束后验证
    save_strategy="epoch",          # 每轮结束后保存模型
    logging_dir="./output/gpt2-finetuned/logs",           # 日志路径
    logging_steps=10,
    learning_rate=5e-5,             # 学习率（GPT类模型通常用2e-5 ~ 5e-5）
    weight_decay=0.01,              # 权重衰减（正则化）
    fp16=False,                      # 是否启用混合精度训练（需GPU支持）
    load_best_model_at_end=True,    # 训练结束后加载最佳模型
    metric_for_best_model="acc",  # 以准确率为判断标准 默认为损失
    greater_is_better=True,  # 准确率越高越好（损失则设为 False， 默认为损失）
)

### 迭代训练

In [21]:
from transformers import DataCollatorForLanguageModeling, DataCollatorWithPadding, DataCollatorForTokenClassification

# 设置日志级别
from transformers import logging 
logging.set_verbosity_info() # INFO级别，默认为WARNING

# 数据填充对象
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 初始化Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,  # 传入数据整理器
    train_dataset=sample_dataset["train"],
    eval_dataset=sample_dataset["validation"],
    compute_metrics=compute_metrics,  # 评估指标
)

[2026-01-30 17:16:38,298] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-01-30 17:16:40,086] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [22]:
# 开始训练
trainer.train()

***** Running training *****
  Num examples = 202,811
  Num Epochs = 5
  Instantaneous batch size per device = 128
  Total train batch size (w. parallel, distributed & accumulation) = 256
  Gradient Accumulation steps = 2
  Total optimization steps = 3,965
  Number of trainable parameters = 85,863,936
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss,Acc
1,2.002700,2.206071,0.027396
2,1.936800,2.143784,0.027492
3,1.905900,2.114187,0.025619
4,1.903500,2.096687,0.027523
5,1.874800,2.090732,0.027707



***** Running Evaluation *****
  Num examples = 50730
  Batch size = 256
Saving model checkpoint to ./output/gpt2-finetuned/models/checkpoint-793
Configuration saved in ./output/gpt2-finetuned/models/checkpoint-793/config.json
Configuration saved in ./output/gpt2-finetuned/models/checkpoint-793/generation_config.json
Model weights saved in ./output/gpt2-finetuned/models/checkpoint-793/model.safetensors
Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
tokenizer config file saved in ./output/gpt2-finetuned/models/checkpoint-793/tokenizer_config.json
Special tokens file saved in ./output/gpt2-finetuned/models/checkpoint-793/special_tokens_map.json

***** Running Evaluation *****
  Num examples = 50730
  Batch size = 256
Saving model checkpoint to ./output/gpt2-finetuned/models/checkpoint-1586
Configuration saved in ./output/gpt2-finetuned/models/checkpoint-1586/config.json
Configuration saved in ./output/gpt2-finetuned/models/checkpoint-1586/generat

TrainOutput(global_step=3965, training_loss=1.9484889175973052, metrics={'train_runtime': 602.4258, 'train_samples_per_second': 1683.286, 'train_steps_per_second': 6.582, 'total_flos': 9148137179904000.0, 'train_loss': 1.9484889175973052, 'epoch': 5.0})

In [23]:
# 模型保存(最终的最优模型保存)
trainer.save_model(os.path.join(training_args.output_dir, "./final-best-model"))

Saving model checkpoint to ./output/gpt2-finetuned/models/./final-best-model
Configuration saved in ./output/gpt2-finetuned/models/./final-best-model/config.json
Configuration saved in ./output/gpt2-finetuned/models/./final-best-model/generation_config.json
Model weights saved in ./output/gpt2-finetuned/models/./final-best-model/model.safetensors
Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
tokenizer config file saved in ./output/gpt2-finetuned/models/./final-best-model/tokenizer_config.json
Special tokens file saved in ./output/gpt2-finetuned/models/./final-best-model/special_tokens_map.json


## 模型推理应用

### 基于pipeline的推理 一

In [24]:
from transformers import pipeline

local_path = "./output/gpt2-finetuned/models/final-best-model"

generator = pipeline(
    'text-generation', 
    model=local_path, 
    tokenizer=EnglishGenTokenizer.from_pretrained(local_path, add_eos_token=False)
)
print(type(generator))
print(generator.model)
print(generator.tokenizer)

loading file vocab.txt
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file tokenizer.json
loading file chat_template.jinja
loading configuration file ./output/gpt2-finetuned/models/final-best-model/config.json
Model config GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 1,
  "embd_pdrop": 0.1,
  "eos_token_id": 1,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": 

<class 'transformers.pipelines.text_generation.TextGenerationPipeline'>
GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(28, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=28, b

In [26]:
generator('h')

[{'generated_text': 'hanka'}]

In [30]:
while True:
    word = input("请输入单词前缀:")

    if word == '1':
        break

    # 生成token id
    result = generator(word)
    print(f"预测结果为:{result[0]['generated_text']}")

请输入单词前缀: he


预测结果为:hemitophoric


请输入单词前缀: he


预测结果为:herring


请输入单词前缀: he


...

请输入单词前缀: he


预测结果为:heatching


请输入单词前缀: he


预测结果为:helloweth


请输入单词前缀: 1
